# SECCIÓN 1 — Instalación y dependencias

En esta sección instalaremos todas las librerías necesarias para:

- análisis de datos
- manipulación de DataFrames
- conexión con Mistral AI
- creación de agentes con LangChain

In [1]:
# Instalación de librerías principales

!pip install -q \
pandas \
numpy \
matplotlib \
seaborn \
mistralai \
langchain \
langchain-community \
langchain-experimental \
langchain-mistralai \
tabulate

# Verificación de instalación

print("Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
Librerías instaladas correctamente


# SECCIÓN 2 — Importaciones

En esta sección importaremos todas las librerías necesarias para:

- manipulación de datos
- visualización
- conexión con Mistral
- creación del agente pandas

In [2]:
# Importaciones principales

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Manejo de tiempo y sistema
import os
import warnings

# Mistral AI
from mistralai.client import Mistral

# LangChain + Mistral
from langchain_mistralai import ChatMistralAI

# Agente pandas
from langchain_experimental.agents.agent_toolkits import (
    create_pandas_dataframe_agent
)

# Configuración visual
warnings.filterwarnings("ignore")

print("Importaciones realizadas correctamente")

Importaciones realizadas correctamente


# SECCIÓN 3 — Configuración de API Key

En esta sección configuraremos nuestra API Key de Mistral AI utilizando Google Colab Secrets.

Esto permite:
- mayor seguridad
- evitar exponer credenciales
- mantener el notebook limpio

In [3]:
# Configuración segura de API Key

from google.colab import userdata

# Obtener API Key desde Colab Secrets
api_key = userdata.get("MistralProyecto")

# Guardar temporalmente en entorno
os.environ["MISTRAL_API_KEY"] = api_key

print("API Key configurada correctamente")

API Key configurada correctamente


# SECCIÓN 4 — Instalación de Mistral y test de conexión

En esta sección:

- configuraremos el cliente de Mistral
- probaremos la conexión con la API
- realizaremos una generación simple de texto

In [4]:
# Configuración de cliente Mistral

# Crear cliente oficial de Mistral
client = Mistral(api_key=api_key)

print("Cliente Mistral configurado correctamente")

# Test simple de generación de texto
response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {
            "role": "user",
            "content": "Explain briefly what Artificial Intelligence is."
        }
    ]
)

# Mostrar respuesta
print("\nRespuesta del modelo:\n")

print(response.choices[0].message.content)

Cliente Mistral configurado correctamente

Respuesta del modelo:

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines, enabling them to perform tasks that typically require human cognition, such as learning, problem-solving, decision-making, and understanding language. AI systems use algorithms and data to analyze information, recognize patterns, and make predictions or decisions with varying degrees of autonomy. Examples include virtual assistants (like Siri or Alexa), recommendation systems (like Netflix or Amazon), and autonomous vehicles. AI can be categorized into **narrow AI** (designed for specific tasks) and **general AI** (hypothetical systems with human-like reasoning).


# SECCIÓN 5 — Carga del dataset CSV

En esta sección:

- cargaremos el archivo CSV
- verificaremos que el dataset se lea correctamente
- almacenaremos el DataFrame principal

In [5]:
# Carga interactiva del dataset

from google.colab import files

# Subir archivo CSV manualmente
uploaded = files.upload()

# Obtener nombre del archivo

nombre_archivo = list(uploaded.keys())[0]

print(f"Archivo cargado: {nombre_archivo}")

# Leer dataset

df = pd.read_csv(nombre_archivo, encoding="latin1")

# Vista previa del dataset

print("\nPrimeras filas del dataset:\n")

df.head()

Saving ventas.csv to ventas (2).csv
Archivo cargado: ventas (2).csv

Primeras filas del dataset:



,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


# SECCIÓN 6 — Normalización del dataset

En esta sección:

- normalizaremos nombres de columnas
- verificaremos valores nulos
- analizaremos tipos de datos
- eliminaremos duplicados
- generaremos estadísticas generales
- guardaremos un nuevo CSV limpio

IMPORTANTE:

A partir de este punto trabajaremos únicamente
con el dataset normalizado.

In [6]:
# Copia de seguridad del dataset original
df_original = df.copy()

# Normalización de nombres de columnas
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

print("Columnas normalizadas:\n")

print(df.columns.tolist())

# Verificación de valores nulos
print("\nValores nulos por columna:\n")

print(df.isnull().sum())

# Eliminación de duplicados
duplicados_antes = df.duplicated().sum()

df = df.drop_duplicates()

duplicados_despues = df.duplicated().sum()

print(f"\nDuplicados eliminados: {duplicados_antes - duplicados_despues}")

# Tipos de datos
print("\nTipos de datos:\n")

print(df.dtypes)

# Estadísticas generales
print("\nEstadísticas generales:\n")

print(df.describe(include="all"))

# Guardar dataset limpio
nombre_csv_limpio = "ventas_limpio.csv"

df.to_csv(nombre_csv_limpio, index=False)

print(f"\nDataset limpio guardado como: {nombre_csv_limpio}")

# Recargar dataset limpio
df = pd.read_csv(nombre_csv_limpio)

print("\nDataset limpio recargado correctamente")

# Vista previa final
df.head()

Columnas normalizadas:

['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'addressline2', 'city', 'state', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize']

Valores nulos por columna:

ordernumber            0
quantityordered        0
priceeach              0
orderlinenumber        0
sales                  0
orderdate              0
status                 0
qtr_id                 0
month_id               0
year_id                0
productline            0
msrp                   0
productcode            0
customername           0
phone                  0
addressline1           0
addressline2        2521
city                   0
state               1486
postalcode            76
country                0
territory           1074
contactlastname        0
contactfirstname       0
dealsize      

,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


# SECCIÓN 7 — Creación del agente pandas

En esta sección:

- configuraremos el modelo Mistral Small
- crearemos un agente pandas con LangChain
- habilitaremos tool-calling
- prepararemos el sistema para responder preguntas sobre el CSV

IMPORTANTE:

El agente utilizará el DataFrame limpio como fuente principal de información.

In [7]:
# Configuración del modelo Mistral
llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=api_key,
    temperature=0
)

print("Modelo Mistral configurado correctamente")


# Prompt de comportamiento del agente
prefix = """
Eres un asistente especializado en análisis de datos.
Trabjas con un DataFrame que tiene estas Columnas:
- ORDERNUMBER: numero de orden
- QUANTITYORDERED: cantidad de productos pedidos
- PRICEEACH: precio por unidad
- ORDERLINENUMBER: numero de linea de la orden
- SALES: ventas totales
- ORDERDATE: fecha de la orden
- STATUS: estado de la orden
- PRODUCTLINE: linea de producto
- CITY: ciudad
- STATE: estado
- POSTALCODE: codigo postal
- COUNTRY: pais
- TERRITORY: territorio
- CONTACTLASTNAME: apellido de contacto
- CONTACTFIRSTNAME: nombre de contacto
- DEALSIZE: tamaño de la orden
Debes:
- responder únicamente utilizando el DataFrame
- responder siempre en español
- evitar inventar información
- realizar cálculos reales usando pandas
- ser preciso y breve
- Si no podes responder, decilo claramente
"""

# Creación del agente pandas
agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True,
    prefix=prefix
)

print("\nAgente pandas creado correctamente")

Modelo Mistral configurado correctamente

Agente pandas creado correctamente


# SECCIÓN 8 — Pruebas y preguntas al agente

En esta sección realizaremos pruebas reales con el agente pandas.

El objetivo es validar que:

- el agente pueda leer el DataFrame
- responda correctamente en español
- utilice pandas internamente
- evite alucinaciones

In [8]:
# Primera prueba simple
pregunta = "¿Cuántas filas y columnas tiene el dataset?"

respuesta = agent.invoke(pregunta)

print("\nRespuesta del agente:\n")

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape'}`


(2823, 25)El dataset tiene **2823 filas** y **25 columnas**.

> Finished chain.

Respuesta del agente:

El dataset tiene **2823 filas** y **25 columnas**.


In [9]:
# Segunda prueba
pregunta = "¿Cuáles son las columnas del dataset?"

respuesta = agent.invoke(pregunta)

print("\nRespuesta del agente:\n")

print(respuesta["output"])



> Entering new AgentExecutor chain...
Las columnas del dataset son:

- ordernumber
- quantityordered
- priceeach
- orderlinenumber
- sales
- orderdate
- status
- qtr_id
- month_id
- year_id
- productline
- msrp
- productcode
- customername
- phone
- addressline1
- addressline2
- city
- state
- postalcode
- country
- territory
- contactlastname
- contactfirstname
- dealsize

> Finished chain.

Respuesta del agente:

Las columnas del dataset son:

- ordernumber
- quantityordered
- priceeach
- orderlinenumber
- sales
- orderdate
- status
- qtr_id
- month_id
- year_id
- productline
- msrp
- productcode
- customername
- phone
- addressline1
- addressline2
- city
- state
- postalcode
- country
- territory
- contactlastname
- contactfirstname
- dealsize


In [10]:
# Tercera prueba
pregunta = "¿Qué país tiene la mayor cantidad de ventas?"

respuesta = agent.invoke(pregunta)

print("\nRespuesta del agente:\n")

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.groupby('COUNTRY')['SALES'].sum().idxmax()"}`


KeyError: 'COUNTRY'
Invoking: `python_repl_ast` with `{'query': "df.groupby('country')['sales'].sum().idxmax()"}`


USAEl país con la mayor cantidad de ventas es **USA**.

> Finished chain.

Respuesta del agente:

El país con la mayor cantidad de ventas es **USA**.


In [11]:
# Cuarta prueba
pregunta = "Muéstrame las primeras 5 filas del dataset"

respuesta = agent.invoke(pregunta)

print("\nRespuesta del agente:\n")

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.head()'}`


   ordernumber  quantityordered  priceeach  orderlinenumber    sales  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
3        10145               45      83.26                6  3746.70   
4        10159               49     100.00               14  5205.27   

         orderdate   status  qtr_id  month_id  year_id  productline  msrp  \
0   2/24/2003 0:00  Shipped       1         2     2003  Motorcycles    95   
1    5/7/2003 0:00  Shipped       2         5     2003  Motorcycles    95   
2    7/1/2003 0:00  Shipped       3         7     2003  Motorcycles    95   
3   8/25/2003 0:00  Shipped       3         8     2003  Motorcycles    95   
4  10/10/2003 0:00  Shipped       4        10     2003  Motorcycles    95   

  p

# SECCIÓN 9 — Instalación de librerías RAG

En esta sección instalaremos las librerías necesarias para construir el sistema RAG.

Estas librerías permitirán:

- crear documentos
- generar embeddings
- almacenar vectores en ChromaDB
- realizar retrieval semántico
- conectar Mistral con LangChain

In [12]:
# Instalación de librerías RAG

!pip install -q \
langchain==0.3.27 \
langchain-core==0.3.72 \
langchain-community==0.3.27 \
langchain-text-splitters==0.3.9 \
langchain-chroma==0.2.5 \
langchain-huggingface==0.3.1 \
chromadb==1.0.15 \
sentence-transformers==5.0.0 \
opentelemetry-api==1.38.0 \
opentelemetry-sdk==1.38.0

# Importaciones principales para RAG

# Documento base
from langchain_core.documents import Document

# División de texto
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings

# Base vectorial ChromaDB
from langchain_chroma import Chroma

# Prompt templates
from langchain_core.prompts import PromptTemplate

# Pipeline RAG
from langchain.chains import RetrievalQA

print("Librerías RAG instaladas correctamente")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistralai 2.4.7 requires opentelemetry-semantic-conventions<0.61,>=0.60b1, but you have opentelemetry-semantic-conventions 0.59b0 which is incompatible.
Librerías RAG instaladas correctamente


# SECCIÓN 10 — Creación de documentos semánticos

En esta sección transformaremos cada fila del CSV
en documentos semánticos compatibles con RAG.

Estos documentos serán utilizados para:

- generar embeddings
- almacenar contexto en ChromaDB
- realizar retrieval semántico

In [13]:
# Creación de documentos semánticos

documentos = []

for _, row in df.iterrows():

    contenido = f"""

    Número de orden: {row.get('ordernumber', '')}
    Cantidad pedida: {row.get('quantityordered', '')}
    Precio por unidad: {row.get('priceeach', '')}
    Línea de orden: {row.get('orderlinenumber', '')}
    Ventas totales: {row.get('sales', '')}
    Fecha de orden: {row.get('orderdate', '')}
    Estado de orden: {row.get('status', '')}
    Línea de producto: {row.get('productline', '')}
    Ciudad: {row.get('city', '')}
    Estado/Provincia: {row.get('state', '')}
    Código postal: {row.get('postalcode', '')}
    País: {row.get('country', '')}
    Territorio: {row.get('territory', '')}
    Apellido de contacto: {row.get('contactlastname', '')}
    Nombre de contacto: {row.get('contactfirstname', '')}
    Tamaño de orden: {row.get('dealsize', '')}

    """

    documentos.append(
        Document(page_content=contenido)
    )

print(f"Documentos creados correctamente: {len(documentos)}")

# Vista previa de un documento
print("\nPrimer documento generado:\n")

print(documentos[0].page_content)

Documentos creados correctamente: 2823

Primer documento generado:


    
    Número de orden: 10107
    Cantidad pedida: 30
    Precio por unidad: 95.7
    Línea de orden: 2
    Ventas totales: 2871.0
    Fecha de orden: 2/24/2003 0:00
    Estado de orden: Shipped
    Línea de producto: Motorcycles
    Ciudad: NYC
    Estado/Provincia: NY
    Código postal: 10022
    País: USA
    Territorio: nan
    Apellido de contacto: Yu
    Nombre de contacto: Kwai
    Tamaño de orden: Small
    
    


# SECCIÓN 11 — División de documentos y embeddings

En esta sección:

- dividiremos los documentos en chunks
- cargaremos el modelo de embeddings
- transformaremos texto en vectores numéricos

Estos embeddings permitirán realizar
búsquedas semánticas dentro del sistema RAG.

In [20]:
# ==========================================
# División de documentos en chunks
# ==========================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documentos)

print(f"Chunks generados: {len(docs)}")


# ==========================================
# Vista previa de un chunk
# ==========================================

print("\nPrimer chunk:\n")

print(docs[0].page_content)


# ==========================================
# Función para embeddings con Mistral
# ==========================================

def generar_embedding(texto):

    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[texto]
    )

    return response.data[0].embedding


# ==========================================
# Prueba de embedding
# ==========================================

texto_prueba = docs[0].page_content

vector_prueba = generar_embedding(texto_prueba)

print(f"\nDimensión del embedding: {len(vector_prueba)}")


# ==========================================
# Vista parcial del vector
# ==========================================

print("\nPrimeros valores del embedding:\n")

print(vector_prueba[:10])

Chunks generados: 2823

Primer chunk:

Número de orden: 10107
    Cantidad pedida: 30
    Precio por unidad: 95.7
    Línea de orden: 2
    Ventas totales: 2871.0
    Fecha de orden: 2/24/2003 0:00
    Estado de orden: Shipped
    Línea de producto: Motorcycles
    Ciudad: NYC
    Estado/Provincia: NY
    Código postal: 10022
    País: USA
    Territorio: nan
    Apellido de contacto: Yu
    Nombre de contacto: Kwai
    Tamaño de orden: Small

Dimensión del embedding: 1024

Primeros valores del embedding:

[-0.0286865234375, 0.00640869140625, 0.0367431640625, 0.01361083984375, 0.055572509765625, 0.0192718505859375, 0.04150390625, 0.001453399658203125, 0.0182647705078125, 0.0032749176025390625]


# SECCIÓN 12 — ChromaDB y sistema RAG

En esta sección:

- crearemos la base vectorial ChromaDB
- configuraremos el retriever
- construiremos el prompt del sistema RAG
- conectaremos Mistral con retrieval
- crearemos el pipeline final de consultas

In [22]:
# ==========================================
# Wrapper optimizado para embeddings Mistral
# ==========================================

import time

class MistralEmbeddingFunction:

    def embed_documents(self, texts):

        embeddings = []

        batch_size = 20

        for i in range(0, len(texts), batch_size):

            batch = texts[i:i + batch_size]

            try:

                response = client.embeddings.create(
                    model="mistral-embed",
                    inputs=batch
                )

                batch_embeddings = [
                    item.embedding
                    for item in response.data
                ]

                embeddings.extend(batch_embeddings)

                print(f"Lote procesado: {i + len(batch)}/{len(texts)}")

                # pausa anti rate-limit
                time.sleep(1)

            except Exception as e:

                print(f"Error en lote: {e}")

                # espera extra si ocurre rate limit
                time.sleep(5)

        return embeddings


    def embed_query(self, text):

        response = client.embeddings.create(
            model="mistral-embed",
            inputs=[text]
        )

        return response.data[0].embedding


# ==========================================
# Inicializar embeddings
# ==========================================

embedding_function = MistralEmbeddingFunction()

print("Embedding function configurada")


# ==========================================
# Crear ChromaDB
# ==========================================

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

print("ChromaDB creada correctamente")


# ==========================================
# Configuración del retriever
# ==========================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever configurado")


# ==========================================
# Prompt del sistema RAG
# ==========================================

prompt_template = """

Eres un asistente especializado en análisis de datos de ventas.

Trabajas con un DataFrame que contiene las siguientes columnas:

- ORDERNUMBER: número de orden
- QUANTITYORDERED: cantidad de productos pedidos
- PRICEEACH: precio por unidad
- ORDERLINENUMBER: número de línea de la orden
- SALES: ventas totales
- ORDERDATE: fecha de la orden
- STATUS: estado de la orden
- PRODUCTLINE: línea de producto
- CITY: ciudad
- STATE: estado
- POSTALCODE: código postal
- COUNTRY: país
- TERRITORY: territorio
- CONTACTLASTNAME: apellido del contacto
- CONTACTFIRSTNAME: nombre del contacto
- DEALSIZE: tamaño de la orden

Debes:
- responder únicamente utilizando el contexto proporcionado
- responder siempre en español
- no inventar información
- ser preciso y breve
- si no encuentras la respuesta, indícalo claramente

Contexto:
{context}

Pregunta:
{question}

Respuesta:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt configurado")


# ==========================================
# Configuración del modelo Mistral
# ==========================================

llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=api_key,
    temperature=0
)

print("LLM configurado")


# ==========================================
# Crear sistema RAG
# ==========================================

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={
        "prompt": PROMPT
    }
)

print("Sistema RAG creado correctamente")

Embedding function configurada
Lote procesado: 20/2823
Lote procesado: 40/2823
Lote procesado: 60/2823
Lote procesado: 80/2823
Lote procesado: 100/2823
Lote procesado: 120/2823
Lote procesado: 140/2823
Lote procesado: 160/2823
Lote procesado: 180/2823
Lote procesado: 200/2823
Lote procesado: 220/2823
Lote procesado: 240/2823
Lote procesado: 260/2823
Lote procesado: 280/2823
Lote procesado: 300/2823
Lote procesado: 320/2823
Lote procesado: 340/2823
Lote procesado: 360/2823
Lote procesado: 380/2823
Lote procesado: 400/2823
Lote procesado: 420/2823
Lote procesado: 440/2823
Lote procesado: 460/2823
Lote procesado: 480/2823
Lote procesado: 500/2823
Lote procesado: 520/2823
Lote procesado: 540/2823
Lote procesado: 560/2823
Lote procesado: 580/2823
Lote procesado: 600/2823
Lote procesado: 620/2823
Lote procesado: 640/2823
Lote procesado: 660/2823
Lote procesado: 680/2823
Lote procesado: 700/2823
Lote procesado: 720/2823
Lote procesado: 740/2823
Lote procesado: 760/2823
Lote procesado: 780/282

# SECCIÓN 13 — Consultas al sistema RAG

En esta sección realizaremos las primeras pruebas
del sistema RAG.

El objetivo es verificar:

- retrieval semántico
- comprensión del contexto
- respuestas en español
- reducción de alucinaciones
- funcionamiento completo del pipeline

In [23]:
# ==========================================
# Primera consulta simple
# ==========================================

pregunta = "¿Cuál es el país con más ventas?"

respuesta = rag_chain.invoke(
    {"query": pregunta}
)

print("\nPregunta:")
print(pregunta)

print("\nRespuesta:")
print(respuesta["result"])


# ==========================================
# Segunda consulta
# ==========================================

pregunta = "¿Qué línea de producto generó más ventas?"

respuesta = rag_chain.invoke(
    {"query": pregunta}
)

print("\nPregunta:")
print(pregunta)

print("\nRespuesta:")
print(respuesta["result"])


# ==========================================
# Tercera consulta
# ==========================================

pregunta = "¿Qué ciudad aparece más veces en las órdenes?"

respuesta = rag_chain.invoke(
    {"query": pregunta}
)

print("\nPregunta:")
print(pregunta)

print("\nRespuesta:")
print(respuesta["result"])


Pregunta:
¿Cuál es el país con más ventas?

Respuesta:
Spain

Pregunta:
¿Qué línea de producto generó más ventas?

Respuesta:
La línea de producto "Trucks and Buses" generó más ventas, con un total de **19,719.98** (suma de las ventas de las órdenes 10126, 10259 y 10424).

Pregunta:
¿Qué ciudad aparece más veces en las órdenes?

Respuesta:
Madrid


In [24]:
# ==========================================
# Chat interactivo RAG
# ==========================================

while True:

    pregunta = input("\nPregunta: ")

    if pregunta.lower() == "salir":
        break

    respuesta = rag_chain.invoke(
        {"query": pregunta}
    )

    print("\nRespuesta:")
    print(respuesta["result"])


Pregunta: Cual fue la mayor venta realizada

Respuesta:
La mayor venta realizada en el contexto proporcionado fue **7083.0** (correspondiente a la línea de orden 5 de la orden 10126).

Pregunta: Cuantos dias tiene una semana?

Respuesta:
No tengo información sobre días en una semana en el contexto proporcionado. La respuesta correcta es **7 días**, pero no está basada en los datos del DataFrame.

Pregunta: salir
